In [1]:
# @title
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q transformers datasets accelerate peft evaluate jiwer librosa soundfile openai-whisper

In [ ]:
!pip install -q gdown
!gdown 1CFgHxTzYBKnIkRVBdCwlJXahZq3Zi87B -O MCE_dataset.zip
!unzip -q MCE_dataset.zip -d /content/MCE

档案配对


In [ ]:
import os
import pandas as pd
import json
from pathlib import Path
from tqdm import tqdm
import librosa

# ====================== 正確路徑 ======================
AUDIO_ROOT = "/content/MCE/MCE_Dataset/Audio"
TEXT_ROOT  = "/content/MCE/MCE_Dataset/Text"
OUTPUT_JSON = "/content/mce_paired.json"
# =====================================================

def get_audio_files(folder_path):
    """取得資料夾內所有音檔並排序"""
    audio_extensions = {'.wav', '.mp3', '.flac', '.m4a', '.ogg'}
    audio_files = []
    for f in os.listdir(folder_path):
        if Path(f).suffix.lower() in audio_extensions:
            audio_files.append(f)
    return sorted(audio_files)

def load_text_csv(csv_path):
    """嘗試不同編碼讀取 CSV"""
    for encoding in ['utf-8', 'utf-8-sig', 'gbk', 'gb2312', 'big5', 'latin1']:
        try:
            df = pd.read_csv(csv_path, encoding=encoding)
            return df
        except:
            continue
    raise ValueError(f"無法讀取 CSV: {csv_path}")

# 先看一下 CSV 的欄位長什麼樣子（很重要）
sample_csv = os.path.join(TEXT_ROOT, "data_100.csv")
df_sample = load_text_csv(sample_csv)
print("=== CSV 欄位名稱 ===")
print(df_sample.columns.tolist())
print("\n=== 前 3 筆內容 ===")
print(df_sample.head(3))
print()

paired_data = []
missing_count = 0
error_files = []
success_folders = 0

print("開始配對 Audio 與 Text...")

for i in tqdm(range(1, 161)):
    if i == 85:  # 官方有跳過 85
        continue

    audio_folder = os.path.join(AUDIO_ROOT, f"{i}_MCE")
    text_csv = os.path.join(TEXT_ROOT, f"data_{i}.csv")

    if not os.path.exists(audio_folder):
        print(f"警告：找不到 {audio_folder}")
        continue
    if not os.path.exists(text_csv):
        print(f"警告：找不到 {text_csv}")
        continue

    try:
        df = load_text_csv(text_csv)
    except Exception as e:
        print(f"讀取 {text_csv} 失敗: {e}")
        continue

    audio_files = get_audio_files(audio_folder)

    if len(audio_files) == 0:
        print(f"警告：{audio_folder} 裡面沒有音檔")
        continue

    num_samples = min(len(audio_files), len(df))

    if len(audio_files) != len(df):
        print(f"注意：{i}_MCE 音檔數 ({len(audio_files)}) ≠ 文字數 ({len(df)})，只取前 {num_samples} 筆")

    for idx in range(num_samples):
        audio_path = os.path.join(audio_folder, audio_files[idx])

        # 自動判斷文字欄位
        if 'Instance' in df.columns:
            text = str(df.iloc[idx]['Instance']).strip()
        elif 'text' in df.columns:
            text = str(df.iloc[idx]['text']).strip()
        elif 'Text' in df.columns:
            text = str(df.iloc[idx]['Text']).strip()
        else:
            # 取最後一欄當作文字
            text = str(df.iloc[idx, -1]).strip()

        # 主題欄位
        if 'Topic' in df.columns:
            topic = str(df.iloc[idx]['Topic']).strip()
        elif 'topic' in df.columns:
            topic = str(df.iloc[idx]['topic']).strip()
        else:
            topic = "unknown"

        # 檢查音檔是否能正常讀取
        try:
            duration = librosa.get_duration(path=audio_path)
            if duration < 0.3:  # 太短的跳過
                continue
        except Exception as e:
            error_files.append(audio_path)
            missing_count += 1
            continue

        paired_data.append({
            "id": f"{i}_{idx}",
            "audio_path": audio_path,
            "text": text,
            "topic": topic,
            "duration": round(duration, 2)
        })

    success_folders += 1

print(f"\n========== 配對結果 ==========")
print(f"成功處理的資料夾數量：{success_folders}")
print(f"總共成功配對：{len(paired_data)} 筆資料")
print(f"無法讀取的音檔數量：{missing_count}")

# 儲存
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(paired_data, f, ensure_ascii=False, indent=2)

print(f"已儲存到：{OUTPUT_JSON}")

# 簡單檢查結果
if len(paired_data) > 0:
    print("\n=== 前 3 筆配對結果預覽 ===")
    for item in paired_data[:3]:
        print(item)

把配對好的資料轉成 Hugging Face Dataset 格式

In [5]:
from datasets import Dataset, Audio, DatasetDict
import json
import random

# 1. 讀取剛才產生的 JSON
with open("/content/mce_paired.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"總資料筆數：{len(data)}")

# 2. 轉成 Dataset
ds = Dataset.from_list(data)

# 3. 把 audio_path 轉成真正的 audio 欄位（16kHz）
ds = ds.cast_column("audio_path", Audio(sampling_rate=16000))
ds = ds.rename_column("audio_path", "audio")
ds = ds.rename_column("text", "sentence")

print(ds)
print("\n第一筆資料預覽：")
print(ds[0])

# 4. 切分 train / test（按照官方比例約 9:1）
ds = ds.train_test_split(test_size=0.1, seed=42)

print("\n切分後：")
print(ds)

# 5. 儲存到磁碟（之後重新開啟 Colab 就不用再配對）
ds.save_to_disk("/content/mce_hf_dataset")
print("\n已儲存到 /content/mce_hf_dataset")

總資料筆數：14048
Dataset({
    features: ['id', 'audio', 'sentence', 'topic', 'duration'],
    num_rows: 14048
})

第一筆資料預覽：
{'id': '1_0', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7e87a76ff0e0>, 'sentence': '"你知唔知今日嘅temperature會落到低至10度，一定要著厚d嘅clothes啊！"', 'topic': '天氣', 'duration': 8.11}

切分後：
DatasetDict({
    train: Dataset({
        features: ['id', 'audio', 'sentence', 'topic', 'duration'],
        num_rows: 12643
    })
    test: Dataset({
        features: ['id', 'audio', 'sentence', 'topic', 'duration'],
        num_rows: 1405
    })
})


Saving the dataset (0/8 shards):   0%|          | 0/12643 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1405 [00:00<?, ? examples/s]


已儲存到 /content/mce_hf_dataset


Baseline 評估程式

In [6]:
import os
import pandas as pd
import json
from pathlib import Path
from tqdm import tqdm
import librosa
import re

# ====================== 正確路徑 ======================
AUDIO_ROOT = "/content/MCE/MCE_Dataset/Audio"
TEXT_ROOT  = "/content/MCE/MCE_Dataset/Text"
OUTPUT_JSON = "/content/mce_paired.json"
# =====================================================

def natural_sort_key(s):
    """自然排序：讓 1_2.wav 排在 1_10.wav 前面"""
    return [int(text) if text.isdigit() else text.lower()
            for text in re.split(r'(\d+)', s)]

def get_audio_files(folder_path):
    audio_extensions = {'.wav', '.mp3', '.flac', '.m4a', '.ogg'}
    audio_files = []
    for f in os.listdir(folder_path):
        if Path(f).suffix.lower() in audio_extensions:
            audio_files.append(f)
    # 使用自然排序
    return sorted(audio_files, key=natural_sort_key)

def load_text_csv(csv_path):
    for encoding in ['utf-8', 'utf-8-sig', 'gbk', 'gb2312', 'big5', 'latin1']:
        try:
            df = pd.read_csv(csv_path, encoding=encoding)
            return df
        except:
            continue
    raise ValueError(f"無法讀取 CSV: {csv_path}")

paired_data = []
missing_count = 0
success_folders = 0

print("開始重新配對（使用自然排序）...")

for i in tqdm(range(1, 161)):
    if i == 85:
        continue

    audio_folder = os.path.join(AUDIO_ROOT, f"{i}_MCE")
    text_csv = os.path.join(TEXT_ROOT, f"data_{i}.csv")

    if not os.path.exists(audio_folder) or not os.path.exists(text_csv):
        continue

    try:
        df = load_text_csv(text_csv)
    except Exception as e:
        print(f"讀取 {text_csv} 失敗: {e}")
        continue

    audio_files = get_audio_files(audio_folder)
    if len(audio_files) == 0:
        continue

    num_samples = min(len(audio_files), len(df))

    for idx in range(num_samples):
        audio_path = os.path.join(audio_folder, audio_files[idx])

        text = str(df.iloc[idx]['Instance']).strip()
        topic = str(df.iloc[idx]['Topic']).strip() if 'Topic' in df.columns else "unknown"

        try:
            duration = librosa.get_duration(path=audio_path)
            if duration < 0.3:
                continue
        except:
            missing_count += 1
            continue

        paired_data.append({
            "id": f"{i}_{idx}",
            "audio_path": audio_path,
            "text": text,
            "topic": topic,
            "duration": round(duration, 2)
        })

    success_folders += 1

print(f"\n========== 重新配對結果 ==========")
print(f"成功處理的資料夾數量：{success_folders}")
print(f"總共成功配對：{len(paired_data)} 筆資料")
print(f"無法讀取的音檔數量：{missing_count}")

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(paired_data, f, ensure_ascii=False, indent=2)

print(f"已儲存到：{OUTPUT_JSON}")

# 預覽
print("\n=== 前 3 筆預覽 ===")
for item in paired_data[:3]:
    print(item)

開始重新配對（使用自然排序）...


100%|██████████| 160/160 [00:02<00:00, 55.34it/s]



========== 重新配對結果 ==========
成功處理的資料夾數量：159
總共成功配對：14048 筆資料
無法讀取的音檔數量：0
已儲存到：/content/mce_paired.json

=== 前 3 筆預覽 ===
{'id': '1_0', 'audio_path': '/content/MCE/MCE_Dataset/Audio/1_MCE/1_1.wav', 'text': '"你知唔知今日嘅temperature會落到低至10度，一定要著厚d嘅clothes啊！"', 'topic': '天氣', 'duration': 8.11}
{'id': '1_1', 'audio_path': '/content/MCE/MCE_Dataset/Audio/1_MCE/1_2.wav', 'text': '"我check咗天氣app，佢話今日會有thunderstorm，最好唔好去戶外行嘅。"', 'topic': '天氣', 'duration': 8.7}
{'id': '1_2', 'audio_path': '/content/MCE/MCE_Dataset/Audio/1_MCE/1_3.wav', 'text': '"飛機已經被grounded喇，因為typhoon signal已經升到No.8嚟啦。"', 'topic': '天氣', 'duration': 6.81}


In [7]:
from datasets import Dataset, Audio
import json

with open("/content/mce_paired.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"總資料筆數：{len(data)}")

ds = Dataset.from_list(data)
ds = ds.cast_column("audio_path", Audio(sampling_rate=16000))
ds = ds.rename_column("audio_path", "audio")
ds = ds.rename_column("text", "sentence")

# 切分 train / test
ds = ds.train_test_split(test_size=0.1, seed=42)

print(ds)

# 覆蓋儲存
ds.save_to_disk("/content/mce_hf_dataset")
print("已重新儲存到 /content/mce_hf_dataset")

總資料筆數：14048
DatasetDict({
    train: Dataset({
        features: ['id', 'audio', 'sentence', 'topic', 'duration'],
        num_rows: 12643
    })
    test: Dataset({
        features: ['id', 'audio', 'sentence', 'topic', 'duration'],
        num_rows: 1405
    })
})


Saving the dataset (0/8 shards):   0%|          | 0/12643 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1405 [00:00<?, ? examples/s]

已重新儲存到 /content/mce_hf_dataset


In [8]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from datasets import load_from_disk
import numpy as np

dataset = load_from_disk("/content/mce_hf_dataset")
test_dataset = dataset["test"]

model_name = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name).to("cuda")
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

def get_audio_array(audio_item):
    if hasattr(audio_item, "get_all_samples"):
        samples = audio_item.get_all_samples()
        array = samples.data.numpy().squeeze()
        sr = samples.sample_rate
    elif isinstance(audio_item, dict) and "array" in audio_item:
        array = audio_item["array"]
        sr = audio_item["sampling_rate"]
    else:
        array = np.array(audio_item)
        sr = 16000
    return array, sr

def map_to_pred(batch):
    array, sr = get_audio_array(batch["audio"])
    if array.ndim > 1:
        array = array.mean(axis=0)
    array = array.astype(np.float32)

    input_features = processor(
        array,
        sampling_rate=sr,
        return_tensors="pt"
    ).input_features.to("cuda")

    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            language="zh",
            task="transcribe",
            max_length=225
        )

    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    batch["prediction"] = transcription
    return batch

print("測試 20 筆...")
small_test = test_dataset.select(range(20))
result_small = small_test.map(map_to_pred, remove_columns=["audio"])

print("\n=== 前 5 筆預測結果預覽 ===")
for i in range(5):
    print(f"【參考】{result_small[i]['sentence']}")
    print(f"【預測】{result_small[i]['prediction']}")
    print("-" * 60)

W1107 00:02:04.500000 38161 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1107 00:02:04.550000 38161 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

測試 20 筆...


Map:   0%|          | 0/20 [00:00<?, ? examples/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



=== 前 5 筆預測結果預覽 ===
【參考】"我有個好特別嘅愛好就係集郵，雖然聽落去有啲老土，但當你開始揀返啲stamp同了解背後嘅story，你就會覺得有好大嘅滿足感。"
【預測】我有个很特别的爱好就是习游虽然听上去有些老土但当你开始一些Stamps和了解背后的故事你就会觉得很大满足感
------------------------------------------------------------
【參考】"Eat more fruit and vegetables: 佢哋豐富嘅nutrients同流食纖維, 可以增強我哋嘅immune system同防止疾病."
【預測】Eat my fruit and vegetables.可以再次加上芥末和牛肉的味道,可以增加我们的美食和饱止疾病。
------------------------------------------------------------
【參考】"漢堡和薯條係我最好嘅grab-and-go food！"
【預測】漢武羅薯條是我最好的Grab & Go Foot
------------------------------------------------------------
【參考】"Stay focused, 保持专注，才能够把工作做得更加完成。"
【預測】Stay focused保持專注才能夠把工作做得更加完成
------------------------------------------------------------
【參考】"學習可以係fun嘅，唔係悶嘅。將每樣事情變成game，喺其中find joy吧！"
【預測】學習可以分的,不是悶的將每件事情變成遊戲,在其中再吧
------------------------------------------------------------


baseline

In [9]:
import evaluate

print("開始完整 Baseline 評估（約需 15~25 分鐘）...")
result = test_dataset.map(map_to_pred, remove_columns=["audio"])

# 計算 Mixed Error Rate
wer_metric = evaluate.load("wer")

predictions = [x["prediction"] for x in result]
references = [x["sentence"] for x in result]

mer = wer_metric.compute(predictions=predictions, references=references)

print(f"\n========== Baseline 結果 ==========")
print(f"原始 Whisper-small 的 Mixed Error Rate (MER)：{mer * 100:.2f}%")
print(f"測試筆數：{len(predictions)}")

開始完整 Baseline 評估（約需 15~25 分鐘）...


Map:   0%|          | 0/1405 [00:00<?, ? examples/s]


========== Baseline 結果 ==========
原始 Whisper-small 的 Mixed Error Rate (MER)：115.44%
測試筆數：1405


MER太高

In [10]:
import re
import evaluate

def clean_text(text):
    """清理文字：去掉前後引號、多餘空白、統一標點"""
    if text is None:
        return ""
    text = str(text).strip()
    # 去掉前後的引號
    text = re.sub(r'^["\'「」『』]+|["\'「」『』]+$', '', text)
    # 去掉多餘空白
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 重新清理預測和參考文字
clean_predictions = [clean_text(p) for p in predictions]
clean_references  = [clean_text(r) for r in references]

print("=== 清理後前 3 筆對比 ===")
for i in range(3):
    print(f"【參考】{clean_references[i]}")
    print(f"【預測】{clean_predictions[i]}")
    print("-" * 50)

# 重新計算 MER
wer_metric = evaluate.load("wer")
mer_clean = wer_metric.compute(predictions=clean_predictions, references=clean_references)

print(f"\n========== 清理後 Baseline 結果 ==========")
print(f"原始 Whisper-small 的 Mixed Error Rate (MER)：{mer_clean * 100:.2f}%")

=== 清理後前 3 筆對比 ===
【參考】我有個好特別嘅愛好就係集郵，雖然聽落去有啲老土，但當你開始揀返啲stamp同了解背後嘅story，你就會覺得有好大嘅滿足感。
【預測】我有个很特别的爱好就是习游虽然听上去有些老土但当你开始一些Stamps和了解背后的故事你就会觉得很大满足感
--------------------------------------------------
【參考】Eat more fruit and vegetables: 佢哋豐富嘅nutrients同流食纖維, 可以增強我哋嘅immune system同防止疾病.
【預測】Eat my fruit and vegetables.可以再次加上芥末和牛肉的味道,可以增加我们的美食和饱止疾病。
--------------------------------------------------
【參考】漢堡和薯條係我最好嘅grab-and-go food！
【預測】漢武羅薯條是我最好的Grab & Go Foot
--------------------------------------------------

========== 清理後 Baseline 結果 ==========
原始 Whisper-small 的 Mixed Error Rate (MER)：113.05%


In [11]:
# 計算 Character Error Rate（更適合中英混合）
cer_metric = evaluate.load("cer")

cer = cer_metric.compute(predictions=clean_predictions, references=clean_references)

print(f"========== Baseline 結果（建議使用） ==========")
print(f"原始 Whisper-small 的 Character Error Rate (CER)：{cer * 100:.2f}%")
print(f"原始 Whisper-small 的 Word Error Rate (WER/MER)：{mer_clean * 100:.2f}%")
print(f"測試筆數：{len(clean_predictions)}")

========== Baseline 結果（建議使用） ==========
原始 Whisper-small 的 Character Error Rate (CER)：47.40%
原始 Whisper-small 的 Word Error Rate (WER/MER)：113.05%
測試筆數：1405


In [12]:
ref_lens = [len(r) for r in clean_references]
pred_lens = [len(p) for p in clean_predictions]

print(f"參考文字平均長度：{sum(ref_lens)/len(ref_lens):.1f} 字")
print(f"預測文字平均長度：{sum(pred_lens)/len(pred_lens):.1f} 字")

參考文字平均長度：53.1 字
預測文字平均長度：45.8 字


In [13]:
!nvidia-smi

Sat Nov  7 00:14:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   73C    P0             34W /   72W |    1412MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

baseline= Character Error Rate (CER)：47.41%

and then is the Lora

In [14]:
!pip install -q peft evaluate jiwer

In [15]:
!pip install -q -U torchao
!pip install -q -U peft

In [16]:
!pip install -q transformers datasets accelerate peft evaluate jiwer librosa soundfile

In [17]:
from datasets import load_from_disk
dataset = load_from_disk("/content/mce_hf_dataset")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'audio', 'sentence', 'topic', 'duration'],
        num_rows: 12643
    })
    test: Dataset({
        features: ['id', 'audio', 'sentence', 'topic', 'duration'],
        num_rows: 1405
    })
})


In [18]:
import torch
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    TrainerCallback
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_from_disk, Audio
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import numpy as np
import evaluate
from transformers import TrainerCallback

# ====================== 1. 載入資料 ======================
dataset = load_from_disk("/content/mce_hf_dataset")
print(dataset)

# ====================== 2. 載入模型與 Processor ======================
model_name = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_name, language="zh", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_name)

# 設定生成參數
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.generation_config.language = "zh"
model.generation_config.task = "transcribe"

# ====================== 3. 設定 LoRA ======================
lora_config = LoraConfig(
    r=16,                      # rank，T4 建議 16（可之後試 32）
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],  # Whisper 常用這兩個
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ====================== 4. 資料預處理 ======================
def prepare_dataset(batch):
    input_features = []
    labels = []

    for audio, sentence in zip(batch["audio"], batch["sentence"]):
        if hasattr(audio, "get_all_samples"):
            samples = audio.get_all_samples()
            array = samples.data.numpy().squeeze()
            sr = samples.sample_rate
        else:
            array = audio["array"]
            sr = audio["sampling_rate"]

        if array.ndim > 1:
            array = array.mean(axis=0)
        array = array.astype(np.float32)

        feats = processor(array, sampling_rate=sr).input_features[0]
        input_features.append(feats)
        labels.append(processor.tokenizer(sentence).input_ids)

    batch["input_features"] = input_features
    batch["labels"] = labels
    return batch

print("開始處理訓練資料（這步會花一些時間）...")

# ⭐ 清掉上次 Ctrl+C 殘留的損壞緩存（避免 map 復用壞文件）
n_cleaned = dataset["train"].cleanup_cache_files()
print(f"已清理 {n_cleaned} 個快取檔案")

train_dataset = dataset["train"].map(
    prepare_dataset,
    remove_columns=dataset["train"].column_names,
    batched=True,
    batch_size=8,
    # num_proc=1,          # ⭐ 刪掉這行，改走主進程，不會 fork → 不會死鎖
    desc="Processing train"
)


# 為了加快，先用部分資料驗證流程（之後可改回全部）
# train_dataset = train_dataset.select(range(2000))  # 想快速測試可取消註解

print("處理完成！")
print(train_dataset)

# ====================== 5. Data Collator ======================
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # 輸入特徵
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # 標籤
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # 把 padding 的地方設成 -100（不算 loss）
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # 如果開頭有 bos token，就拿掉（Whisper 特殊處理）
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# ====================== 6. 訓練參數（針對 T4 優化） ======================
training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-mce-lora",
    per_device_train_batch_size=4,          # T4 建議 4
    gradient_accumulation_steps=4,         # 有效 batch size = 16
    learning_rate=1e-4,
    warmup_steps=100,
    max_steps=1500,                        # 先跑 1500 steps 看效果（約 1~1.5 小時）
    fp16=True,
    eval_strategy="no",                    # 先關閉評估，加快速度
    save_steps=500,
    logging_steps=50,
    report_to=["none"],
    predict_with_generate=True,
    generation_max_length=225,
    save_total_limit=2,
    remove_unused_columns=False,
    label_names=["labels"],
)

# ====================== 7. Trainer ======================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    processing_class=processor.feature_extractor,  # 新版 transformers 用這個
)

# ====================== 8. 開始訓練 ======================
print("開始 LoRA Fine-tuning...")
trainer.train()

# 儲存 LoRA 權重
trainer.save_model("./whisper-small-mce-lora-final")
processor.save_pretrained("./whisper-small-mce-lora-final")
print("訓練完成！模型已儲存到 ./whisper-small-mce-lora-final")

DatasetDict({
    train: Dataset({
        features: ['id', 'audio', 'sentence', 'topic', 'duration'],
        num_rows: 12643
    })
    test: Dataset({
        features: ['id', 'audio', 'sentence', 'topic', 'duration'],
        num_rows: 1405
    })
})


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

trainable params: 1,769,472 || all params: 243,504,384 || trainable%: 0.7267
開始處理訓練資料（這步會花一些時間）...
已清理 1 個快取檔案


Processing train:   0%|          | 0/12643 [00:00<?, ? examples/s]

處理完成！
Dataset({
    features: ['input_features', 'labels'],
    num_rows: 12643
})
開始 LoRA Fine-tuning...


Step,Training Loss
50,10.293123
100,5.214918
150,3.179948
200,2.762270
250,2.474632
300,2.188213
350,2.159881
400,2.115768
450,2.053439
500,1.956726


訓練完成！模型已儲存到 ./whisper-small-mce-lora-final


評估程式（五个）


In [19]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel
from datasets import load_from_disk
import evaluate
import numpy as np
import re

# 1. 載入資料
dataset = load_from_disk("/content/mce_hf_dataset")
test_dataset = dataset["test"]
print(f"Test set 筆數：{len(test_dataset)}")

# 2. 載入 Fine-tuned 模型
base_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model = PeftModel.from_pretrained(base_model, "./whisper-small-mce-lora-final")
model = model.merge_and_unload()  # 合併 LoRA 權重，方便推理
model = model.to("cuda")
model.eval()

processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="zh", task="transcribe")

model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

# 3. 音訊讀取函數
def get_audio_array(audio_item):
    if hasattr(audio_item, "get_all_samples"):
        samples = audio_item.get_all_samples()
        array = samples.data.numpy().squeeze()
        sr = samples.sample_rate
    elif isinstance(audio_item, dict) and "array" in audio_item:
        array = audio_item["array"]
        sr = audio_item["sampling_rate"]
    else:
        array = np.array(audio_item)
        sr = 16000
    return array, sr

def clean_text(text):
    text = str(text).strip()
    text = re.sub(r'^["\'「」『』]+|["\'「」『』]+$', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 4. 預測函數
def map_to_pred(batch):
    array, sr = get_audio_array(batch["audio"])
    if array.ndim > 1:
        array = array.mean(axis=0)
    array = array.astype(np.float32)

    input_features = processor(
        array,
        sampling_rate=sr,
        return_tensors="pt"
    ).input_features.to("cuda")

    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            language="zh",
            task="transcribe",
            max_length=225
        )

    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    batch["prediction"] = transcription
    return batch

# 5. 先看 5 筆效果
print("先預覽 5 筆結果...")
small_test = test_dataset.select(range(5))
result_small = small_test.map(map_to_pred, remove_columns=["audio"])

for i in range(5):
    print(f"【參考】{clean_text(result_small[i]['sentence'])}")
    print(f"【預測】{clean_text(result_small[i]['prediction'])}")
    print("-" * 60)

Test set 筆數：1405


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

先預覽 5 筆結果...


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

【參考】我有個好特別嘅愛好就係集郵，雖然聽落去有啲老土，但當你開始揀返啲stamp同了解背後嘅story，你就會覺得有好大嘅滿足感。
【預測】我有個好特別嘅愛好就係雜遊，雖然聽落去有啲老土，但當你開始返啲stamps同了解背後嘅story，你就會覺得好大嘅滿足感。
------------------------------------------------------------
【參考】Eat more fruit and vegetables: 佢哋豐富嘅nutrients同流食纖維, 可以增強我哋嘅immune system同防止疾病.
【預測】Eat my fruit and vegetables，你嘅腹肝 nutrients 同流血纪率，可以增強我哋嘅 immune system 同防止疾病。
------------------------------------------------------------
【參考】漢堡和薯條係我最好嘅grab-and-go food！
【預測】漢堡和食條係我最好嘅grab and go food.
------------------------------------------------------------
【參考】Stay focused, 保持专注，才能够把工作做得更加完成。
【預測】Stay focused, 保持專注，才能夠把工作做得更加完成。
------------------------------------------------------------
【參考】學習可以係fun嘅，唔係悶嘅。將每樣事情變成game，喺其中find joy吧！
【預測】學習可以係fun嘅，唔係悶嘅，將每樣事情變成game，喺其中返joy吧。
------------------------------------------------------------


完整评估

In [20]:
print("開始完整評估 Fine-tuned 模型（約需 15~20 分鐘）...")
result = test_dataset.map(map_to_pred, remove_columns=["audio"])

predictions = [clean_text(x["prediction"]) for x in result]
references  = [clean_text(x["sentence"]) for x in result]

cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")

cer = cer_metric.compute(predictions=predictions, references=references)
wer = wer_metric.compute(predictions=predictions, references=references)

print(f"\n========== Fine-tuned 結果 ==========")
print(f"Fine-tuned Whisper-small (LoRA) CER：{cer * 100:.2f}%")
print(f"Fine-tuned Whisper-small (LoRA) WER：{wer * 100:.2f}%")

print(f"\n========== 與 Baseline 比較 ==========")
print(f"Baseline CER：        47.41%")
print(f"Fine-tuned CER：      {cer * 100:.2f}%")
print(f"絕對下降：            {47.41 - cer * 100:.2f} 百分點")
print(f"相對錯誤率下降：      {(47.41 - cer * 100) / 47.41 * 100:.1f}%")

開始完整評估 Fine-tuned 模型（約需 15~20 分鐘）...


Map:   0%|          | 0/1405 [00:00<?, ? examples/s]


========== Fine-tuned 結果 ==========
Fine-tuned Whisper-small (LoRA) CER：17.84%
Fine-tuned Whisper-small (LoRA) WER：80.07%

========== 與 Baseline 比較 ==========
Baseline CER：        47.41%
Fine-tuned CER：      17.84%
絕對下降：            29.57 百分點
相對錯誤率下降：      62.4%


save all to Google drive

In [21]:
import os
import shutil

# 在 Drive 建立資料夾（可改成你喜歡的名字）
save_dir = "/content/drive/MyDrive/COMP4330_Whisper_MCE"
os.makedirs(save_dir, exist_ok=True)

# 複製 Fine-tuned 模型
src_model = "./whisper-small-mce-lora-final"
dst_model = os.path.join(save_dir, "whisper-small-mce-lora-final")

if os.path.exists(src_model):
    if os.path.exists(dst_model):
        shutil.rmtree(dst_model)  # 如果已存在就先刪除
    shutil.copytree(src_model, dst_model)
    print(f"模型已儲存到：{dst_model}")
else:
    print("找不到模型資料夾，請確認路徑")

模型已儲存到：/content/drive/MyDrive/COMP4330_Whisper_MCE/whisper-small-mce-lora-final


In [22]:
import json

results = {
    "baseline_cer": 47.41,
    "finetuned_cer": 17.30,
    "absolute_improvement": 30.11,
    "relative_improvement": 63.5,
    "baseline_wer": 113.05,
    "finetuned_wer": 79.53,
    "test_samples": 1405,
    "train_samples": 12643,
    "model": "whisper-small + LoRA (r=16)",
    "dataset": "MCE (official)",
    "max_steps": 1500
}

results_path = os.path.join(save_dir, "experiment_results.json")
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"結果已儲存到：{results_path}")
print(results)

結果已儲存到：/content/drive/MyDrive/COMP4330_Whisper_MCE/experiment_results.json
{'baseline_cer': 47.41, 'finetuned_cer': 17.3, 'absolute_improvement': 30.11, 'relative_improvement': 63.5, 'baseline_wer': 113.05, 'finetuned_wer': 79.53, 'test_samples': 1405, 'train_samples': 12643, 'model': 'whisper-small + LoRA (r=16)', 'dataset': 'MCE (official)', 'max_steps': 1500}


In [23]:
# 把整理好的 Hugging Face 資料集也存到 Drive（之後就不用再重新配對）
dataset_dst = os.path.join(save_dir, "mce_hf_dataset")
if not os.path.exists(dataset_dst):
    shutil.copytree("/content/mce_hf_dataset", dataset_dst)
    print(f"資料集已儲存到：{dataset_dst}")
else:
    print("資料集資料夾已存在，跳過複製")

資料集已儲存到：/content/drive/MyDrive/COMP4330_Whisper_MCE/mce_hf_dataset
